# A/B-test Simulation

### 1. Defining Input Parameters

In [19]:
# Defining Input Parameters

import numpy as np
import pandas as pd
import json

# Load Input Data from Test Design
with open("../outputs/input_data.json", "r") as f:
    params = json.load(f)

# Simulation Input Parameters
p_control = params["baseline_cr"]
p_treatment = params["treatment_cr"]
n_per_group = params["sample_size_per_group"]
n_total = params["total_sample_size"]
days = params["estimated_days"]

np.random.seed(42)

### 2. Simulate Users

In [20]:
# Getting Outcomes by Group
control = np.random.binomial(1, p_control, n_per_group)
treatment = np.random.binomial(1, p_treatment, n_per_group)

# Concatenate into a single row
outcomes = np.concatenate([control, treatment])

# Getting Groups
groups = ['control'] * n_per_group + ['treatment'] * n_per_group

# Convert to DataFrame
sessions_df = pd.DataFrame({
    "group_name": groups,
    "conversion": outcomes
})

sessions_df.head()

,group_name,conversion
0,control,0
1,control,1
2,control,0
3,control,0
4,control,0


### 3. Check Observed Results

In [21]:
# Observed Rates

observed_rates = sessions_df.groupby('group_name')['conversion'].mean()
observed_rates

group_name
control      0.049822
treatment    0.054248
Name: conversion, dtype: float64

In [22]:
# Observed Lift

cr_control = observed_rates['control']
cr_treatment = observed_rates['treatment']

lift = (cr_treatment - cr_control) / cr_control
lift

np.float64(0.08882907133243603)

### 4. Saving Experiment Check Results

In [23]:
# Collecting and saving A/B-test Results

import json

params = {
    "cr_control": cr_control,
    "cr_treatment": cr_treatment,
    "sample_size_per_group": n_per_group,
    "total_sample_size": n_total,
    "days": days,
    "relative_lift": lift
}

with open("../outputs/experiment_results.json", "w") as f:
    json.dump(params, f, indent=2)


with open("../outputs/experiment_results.json", "r") as f:
    params = json.load(f)

params

{'cr_control': 0.04982230268892912,
 'cr_treatment': 0.054247971568430225,
 'sample_size_per_group': 29826,
 'total_sample_size': 59652,
 'days': 14,
 'relative_lift': 0.08882907133243603}

## 5. Writing Simulation to Database

In [24]:
# Writing simulation results table into database, analytics schema

# Connecting to db
from src.db import get_engine

engine = get_engine()

sessions_df.to_sql(
    name="ab_test_simulation_v1",
    con=engine,
    schema="analytics",
    if_exists="replace",
    index=True
)

652